## Loading Dataset

In [ ]:
import csv
import pandas as pd
import re
import unicodedata

PATH = "spotify_dataset.csv"

with open(PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    header_line = f.readline().rstrip("\n")

raw_header = next(csv.reader([header_line], delimiter=",", quotechar='"', escapechar="\\"))
expected_cols = len(raw_header)

def clean_col(c: str) -> str:
    c = str(c).strip()
    if c.startswith('"') and c.endswith('"') and len(c) >= 2:
        c = c[1:-1].strip()
    return c

header = [clean_col(c) for c in raw_header]

good_rows = []
bad_count = 0
with open(PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f, delimiter=",", quotechar='"', escapechar="\\")
    _ = next(reader)  
    for line_no, row in enumerate(reader, start=2):
        if len(row) == expected_cols:
            good_rows.append(row)
        else:
            bad_count += 1

df = pd.DataFrame(good_rows, columns=header)

print("Loaded shape:", df.shape)
print("Bad rows dropped:", bad_count)
print("Cleaned columns:", list(df.columns))
display(df.head(5))

Loaded shape: (12890338, 4)
Bad rows dropped: 11516
Cleaned columns: ['user_id', 'artistname', 'trackname', 'playlistname']


,user_id,artistname,trackname,playlistname
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010
2,9cc0cfd4d7d7885102480dd99e7a90d6,Tiffany Page,7 Years Too Late,HARD ROCK 2010
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010
4,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010


## Normalize artist + track names

In [3]:
USER_COL = "user_id"
ARTIST_COL = "artistname"
TRACK_COL = "trackname"
PLAYLIST_COL = "playlistname"

missing = [c for c in [USER_COL, ARTIST_COL, TRACK_COL, PLAYLIST_COL] if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}\nAvailable: {list(df.columns)}")

def normalize_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s)

    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))

    s = s.lower().strip()
    s = s.replace("&", " and ")
    s = re.sub(r"[^a-z0-9\s]", " ", s)   
    s = re.sub(r"\s+", " ", s).strip()  
    return s

df[ARTIST_COL] = df[ARTIST_COL].fillna("")
df[TRACK_COL] = df[TRACK_COL].fillna("")
df[PLAYLIST_COL] = df[PLAYLIST_COL].fillna("")
df[USER_COL] = df[USER_COL].fillna("")

df["artist_norm"] = df[ARTIST_COL].map(normalize_text)
df["track_norm"]  = df[TRACK_COL].map(normalize_text)

df["song_key"] = df["artist_norm"] + " — " + df["track_norm"]


## Remove duplicates

In [4]:
df["playlist_key"] = df[USER_COL].astype(str) + " :: " + df[PLAYLIST_COL].map(normalize_text)

before = len(df)
df = df.drop_duplicates(subset=["playlist_key", "song_key"], keep="first")
print("Removed duplicate (playlist, song) rows:", before - len(df))

print("Final shape:", df.shape)
display(df[[USER_COL, PLAYLIST_COL, ARTIST_COL, TRACK_COL, "artist_norm", "track_norm"]].head(10))

Removed duplicate (playlist, song) rows: 45937
Final shape: (12844401, 8)


,user_id,playlistname,artistname,trackname,artist_norm,track_norm
0,9cc0cfd4d7d7885102480dd99e7a90d6,HARD ROCK 2010,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,elvis costello,the angels wanna wear my red shoes
1,9cc0cfd4d7d7885102480dd99e7a90d6,HARD ROCK 2010,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",elvis costello and the attractions,what s so funny bout peace love and understanding
2,9cc0cfd4d7d7885102480dd99e7a90d6,HARD ROCK 2010,Tiffany Page,7 Years Too Late,tiffany page,7 years too late
3,9cc0cfd4d7d7885102480dd99e7a90d6,HARD ROCK 2010,Elvis Costello & The Attractions,Accidents Will Happen,elvis costello and the attractions,accidents will happen
4,9cc0cfd4d7d7885102480dd99e7a90d6,HARD ROCK 2010,Elvis Costello,Alison,elvis costello,alison
5,9cc0cfd4d7d7885102480dd99e7a90d6,HARD ROCK 2010,Lissie,All Be Okay,lissie,all be okay
6,9cc0cfd4d7d7885102480dd99e7a90d6,HARD ROCK 2010,Paul McCartney,Band On The Run,paul mccartney,band on the run
7,9cc0cfd4d7d7885102480dd99e7a90d6,HARD ROCK 2010,Joe Echo,Beautiful,joe echo,beautiful
8,9cc0cfd4d7d7885102480dd99e7a90d6,HARD ROCK 2010,Paul McCartney,"Blackbird - Live at CitiField, NYC - Digital A...",paul mccartney,blackbird live at citifield nyc digital audio
9,9cc0cfd4d7d7885102480dd99e7a90d6,HARD ROCK 2010,Lissie,Bright Side,lissie,bright side
